# InspectEscrow — AI E2E FINAL

**목적:** 지금까지 동결된 InspectEscrow AI/Policy 결과를 바꾸지 않고, 남은 E2E 작업을 한 노트북에서 마무리한다.

이 노트북은 다음 순서로 구성된다.

1. Frozen baseline 검증
2. Supplier/Buyer trade-specific specimen + evidence 준비
3. Phase-1 Evidence Package v1 생성
4. Supplier/Buyer EIP-712 **unsigned** StationAttestation 생성
5. Replay wrapper + unsigned ReplayAttestation 준비
6. Inspector decision template 생성
7. Blockchain handoff + Final QA + ZIP 패키징

## 중요한 실행 규칙

- **모델 재학습/threshold 변경 없음.**
- **Private key를 이 노트북에 넣지 않는다.**
- Supplier/Buyer Evidence Package는 `issuedAt/validUntil`을 포함하므로 **각 실제 서명/제출 직전에 JIT 셀을 실행**한다.
- `proposeTrade()` / `submitInspection()` 등 실제 트랜잭션은 이 노트북이 보내지 않는다.
- `NORMAL` / `PRE_SHIPMENT` 시나리오는 frozen DISPUTE tradeId에 섞지 않는다.
- Public MVTec dataset simulation이므로 실제 physical QR/custody/same-item proof를 주장하지 않는다.

### Frozen meaningful demo
- Supplier: `demo/good/000.png` → PASS
- Buyer: `demo/scratch/022.png` → REVIEW_REQUIRED
- Expected path: `SUPPLIER PASS → BUYER REVIEW → DISPUTED_PRIMARY`

> **셀 1~5는 static preparation.**  
> **Supplier JIT / Buyer JIT / Replay JIT 셀은 실제 서명 직전에 각각 실행한다.**

### Frozen TradeTerm 추가 확인
- `policyVersion = 1` (`uint32`)
- normal GitHub에는 100MB 초과 모델 파일을 직접 포함하지 않는다.


In [ ]:
# ============================================================
# CELL 1 — ENVIRONMENT / IMPORTS
# ============================================================

import sys, subprocess, importlib.util
from pathlib import Path

def ensure(pkg, import_name=None):
    name = import_name or pkg.replace("-", "_")
    if importlib.util.find_spec(name) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

ensure("rfc8785")
ensure("eth-hash", "eth_hash")
ensure("pycryptodome", "Crypto")  # eth-hash backend required for keccak
ensure("eth-abi", "eth_abi")
ensure("jsonschema")
ensure("Pillow", "PIL")

import io
import os
import re
import json
import copy
import time
import secrets
import shutil
import hashlib
import zipfile
import urllib.request
import urllib.error
from decimal import Decimal, ROUND_HALF_UP
from datetime import datetime, timezone
from collections import Counter

import rfc8785
from eth_hash.auto import keccak
from eth_abi import encode
from jsonschema import Draft202012Validator
from PIL import Image

# Colab Drive
try:
    from google.colab import drive
    if not Path("/content/drive/MyDrive").exists():
        drive.mount("/content/drive")
except Exception:
    pass

print("✅ Environment ready")

In [ ]:
# ============================================================
# CELL 2 — FROZEN BASELINE + PATHS + HELPERS
# ============================================================

V2_DIR = Path("/content/drive/MyDrive/MeTal_Dataset/InspectEscrow_AI_MODEL_V2")
DATASET_ZIP = Path("/content/drive/MyDrive/MeTal_Dataset/InspectEscrow_metal_nut_FINAL_DATASET.zip")

POLICY_DIR = V2_DIR / "POLICY"
CANONICAL_DIR = V2_DIR / "CANONICAL_AI_EVIDENCE"
FINAL_DEMO_EVIDENCE_DIR = V2_DIR / "FINAL_DEMO_EVIDENCE"
REPLAY_DIR = V2_DIR / "REPLAY"

DEMO_ROOT = V2_DIR / "ONCHAIN_DEMO" / "DISPUTE"
SUPPLIER_DIR = DEMO_ROOT / "SUPPLIER_ATTEMPT_1"
BUYER_DIR = DEMO_ROOT / "BUYER_ATTEMPT_1"
REPLAY_OUT_DIR = DEMO_ROOT / "REPLAY_PRIMARY"
HANDOFF_DIR = V2_DIR / "FINAL_AI_BLOCKCHAIN_HANDOFF"

for d in [POLICY_DIR, DEMO_ROOT, SUPPLIER_DIR, BUYER_DIR, REPLAY_OUT_DIR, HANDOFF_DIR]:
    d.mkdir(parents=True, exist_ok=True)

INVENTORY_V11 = POLICY_DIR / "ACTIVE_POLICY_TRADE_INPUT_INVENTORY_V11.json"
SCENARIO_PATH = POLICY_DIR / "MEANINGFUL_ONCHAIN_DEMO_SCENARIO_V1.json"
POLICY_PATH = POLICY_DIR / "INSPECTION_POLICY_V1_1_FINAL.json"
CAPTURE_PROFILE_PATH = POLICY_DIR / "DEMO_CAPTURE_PROFILE_V1.json"
REQUIRED_VIEW_SET_PATH = POLICY_DIR / "DEMO_REQUIRED_VIEW_SET_V1.json"

# Frozen values
CHAIN_ID = 11155111
CONTRACT = "0x270e3E8a53dC50A04f0d83b5b4ae9416EB42fFa9"
POLICY_VERSION = 1

TRADE_ID = "0xe021917ece6af29be9a02300cccfe293f484ba685721f49550a98da96d64c94c"
POLICY_HASH = "0x85cd60bd89d1c82d39a313443723997283d79a46ec415c4c1064088ffe3f9d11"
MODEL_BUNDLE_HASH = "0x64f3359a4e9a276a5f0abef0e5305f42b513dbae1ac5698cb08c9e066962383e"
CAPTURE_PROFILE_HASH = "0x3ea93a717cf8570d9b5c0662b1d27bb4a882651784342013786266b269db96da"
PRODUCT_SPEC_HASH = "0x68e3eb25a12d6521ae2aa37735ffad831810f6a05268d940b33a3bae6915d4e7"
REQUIRED_VIEW_SET_HASH = "0x5db9617738ccb9a0b2811e49716ad46cb24474d56a9cd55867df831a01c4abc3"
SCORE_SCHEMA_ID = "0xcf1476a2ae70ac5fad6ba79a7f55bab4b4c4973dd892d65de9a887bf08d1f1f8"

THRESHOLD_SCALED = 57_904_037
SCORE_SCALE_EXPONENT = 6
SCORE_MIN_SCALED = 0
SCORE_MAX_SCALED = 2**128 - 1
REPLAY_TOLERANCE_SCALED = 222
ZERO_BYTES32 = "0x" + "00" * 32

SUPPLIER_SOURCE = "demo/good/000.png"
BUYER_SOURCE = "demo/scratch/022.png"
SUPPLIER_SCORE_SCALED = 55_442_036
BUYER_SCORE_SCALED = 60_138_172

# Solidity enums used by deployed contract
ROLE_SUPPLIER = 0
ROLE_BUYER = 1
VALIDITY_ADMISSIBLE = 0
INVALID_REASON_NONE = 0
INSPECTOR_STAGE_PRIMARY = 0
SETTLEMENT_RELEASE_TO_SUPPLIER = 1
SETTLEMENT_REFUND_TO_BUYER = 2

def load_json(path):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def save_json(path, obj):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, sort_keys=True)

def sha256_bytes(data: bytes) -> str:
    return hashlib.sha256(data).hexdigest()

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def keccak_bytes(data: bytes) -> str:
    return "0x" + keccak(data).hex()

def keccak_file(path: Path) -> str:
    return keccak_bytes(path.read_bytes())

def jcs_bytes(obj) -> bytes:
    return rfc8785.dumps(obj)

def jcs_keccak(obj) -> str:
    return keccak_bytes(jcs_bytes(obj))

def utc_iso(ts: int) -> str:
    return datetime.fromtimestamp(ts, timezone.utc).isoformat().replace("+00:00", "Z")

def is_bytes32(v):
    return isinstance(v, str) and re.fullmatch(r"0x[0-9a-fA-F]{64}", v) is not None

def is_address(v):
    return isinstance(v, str) and re.fullmatch(r"0x[0-9a-fA-F]{40}", v) is not None

def source_bytes_from_zip(relative_path: str):
    assert DATASET_ZIP.exists(), DATASET_ZIP
    with zipfile.ZipFile(DATASET_ZIP, "r") as z:
        matches = [
            n for n in z.namelist()
            if n.replace("\\", "/") == relative_path
            or n.replace("\\", "/").endswith("/" + relative_path)
        ]
        assert len(matches) == 1, f"{relative_path}: ZIP matches={matches}"
        member = matches[0]
        return member, z.read(member)

def decimal_to_scaled(raw_string: str, exponent=6):
    return int(
        (Decimal(raw_string) * Decimal(10**exponent))
        .quantize(Decimal("1"), rounding=ROUND_HALF_UP)
    )

def flatten(obj, prefix="$"):
    out = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            out.extend(flatten(v, prefix + "." + str(k)))
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            out.extend(flatten(v, prefix + f"[{i}]"))
    else:
        out.append((prefix, obj))
    return out

print("✅ Frozen constants/helpers loaded")

In [ ]:
# ============================================================
# CELL 3 — PHASE-1 EVIDENCE PACKAGE v1 SCHEMA
# Exact schema embedded to keep notebook self-contained.
# ============================================================

EVIDENCE_SCHEMA = {
  "$schema": "https://json-schema.org/draft/2020-12/schema",
  "$id": "https://inspectescrow.example/schemas/evidence-package-v1.schema.json",
  "title": "InspectEscrow Evidence Package v1",
  "type": "object",
  "additionalProperties": False,
  "required": ["schema", "schemaVersion", "packageId", "tradeId", "inspectionRole", "attempt", "sessionNonce", "policyHash", "modelBundleHash", "captureProfileHash", "specimen", "files", "captureQa", "score", "station", "createdAt"],
  "properties": {
    "schema": {"const": "inspectescrow.evidence-package"},
    "schemaVersion": {"const": "1.0.0"},
    "packageId": {"type": "string", "minLength": 1},
    "tradeId": {"$ref": "#/$defs/bytes32"},
    "inspectionRole": {"enum": ["SUPPLIER", "BUYER", "INSPECTOR_REPLAY"]},
    "attempt": {"type": "integer", "minimum": 1, "maximum": 255},
    "sessionNonce": {"$ref": "#/$defs/bytes32"},
    "policyHash": {"$ref": "#/$defs/bytes32"},
    "modelBundleHash": {"$ref": "#/$defs/bytes32"},
    "captureProfileHash": {"$ref": "#/$defs/bytes32"},
    "specimen": {
      "type": "object",
      "additionalProperties": False,
      "required": ["specimenId", "lotId", "tradeBound", "overviewFileLogicalName", "requiredViewSetHash", "specimenManifestHash", "physicalSubstitutionNotEliminated"],
      "properties": {
        "specimenId": {"type": "string", "minLength": 1},
        "lotId": {"type": "string", "minLength": 1},
        "tradeBound": {"const": True},
        "overviewFileLogicalName": {"type": "string", "minLength": 1},
        "requiredViewSetHash": {"$ref": "#/$defs/bytes32"},
        "specimenManifestHash": {"$ref": "#/$defs/bytes32"},
        "physicalSubstitutionNotEliminated": {"const": True}
      }
    },
    "files": {
      "type": "array",
      "minItems": 1,
      "items": {
        "type": "object",
        "additionalProperties": False,
        "required": ["logicalName", "viewId", "mediaType", "byteLength", "hashAlgorithm", "digest", "isOriginal", "isOverview"],
        "properties": {
          "logicalName": {"type": "string", "minLength": 1},
          "viewId": {"type": "string", "minLength": 1},
          "mediaType": {"type": "string", "minLength": 1},
          "byteLength": {"type": "integer", "minimum": 1},
          "hashAlgorithm": {"const": "keccak256"},
          "digest": {"$ref": "#/$defs/bytes32"},
          "isOriginal": {"type": "boolean"},
          "isOverview": {"type": "boolean"}
        }
      }
    },
    "captureQa": {
      "type": "object",
      "required": ["inspectionValidity", "invalidReason", "profileConforming", "requiredViewsPresent", "focusPass", "blurPass", "saturationPass"],
      "properties": {
        "inspectionValidity": {"enum": ["ADMISSIBLE", "INVALID"]},
        "invalidReason": {"enum": ["NONE", "REQUIRED_VIEW_MISSING", "IMAGE_QA_FAILED", "CAPTURE_PROFILE_NONCONFORMING", "SPECIMEN_EVIDENCE_INSUFFICIENT", "RECAPTURE_RULE_VIOLATION", "EVIDENCE_PACKAGE_INCOMPLETE", "OTHER_POLICY_DEFINED"]},
        "profileConforming": {"type": "boolean"},
        "requiredViewsPresent": {"type": "boolean"},
        "focusPass": {"type": "boolean"},
        "blurPass": {"type": "boolean"},
        "saturationPass": {"type": "boolean"}
      }
    },
    "score": {
      "type": "object",
      "additionalProperties": False,
      "required": ["scoreSchemaId", "scaleExponent", "perViewScaled", "aggregateScoreScaled", "stationVerdictAuthoritative"],
      "properties": {
        "scoreSchemaId": {"type": "string", "minLength": 1},
        "scaleExponent": {"type": "integer", "minimum": 0, "maximum": 18},
        "perViewScaled": {"type": "array", "items": {"type": "integer", "minimum": 0}},
        "aggregateScoreScaled": {"type": "integer", "minimum": 0},
        "stationVerdictAuthoritative": {"const": False}
      }
    },
    "station": {
      "type": "object",
      "additionalProperties": False,
      "required": ["signer", "signatureScheme", "attestationDigest", "issuedAt", "validUntil"],
      "properties": {
        "signer": {"$ref": "#/$defs/address"},
        "signatureScheme": {"const": "EIP712"},
        "attestationDigest": {"$ref": "#/$defs/bytes32"},
        "issuedAt": {"type": "integer", "minimum": 0},
        "validUntil": {"type": "integer", "minimum": 0}
      }
    },
    "createdAt": {"type": "string", "format": "date-time"},
    "heatmapFiles": {"type": "array", "items": {"$ref": "#/$defs/fileRef"}},
    "reportFiles": {"type": "array", "items": {"$ref": "#/$defs/fileRef"}}
  },
  "$defs": {
    "bytes32": {"type": "string", "pattern": "^0x[0-9a-fA-F]{64}$"},
    "address": {"type": "string", "pattern": "^0x[0-9a-fA-F]{40}$"},
    "fileRef": {
      "type": "object",
      "required": ["logicalName", "mediaType", "byteLength", "digest"],
      "properties": {
        "logicalName": {"type": "string"},
        "mediaType": {"type": "string"},
        "byteLength": {"type": "integer", "minimum": 1},
        "digest": {"$ref": "#/$defs/bytes32"}
      }
    }
  }
}

EVIDENCE_VALIDATOR = Draft202012Validator(EVIDENCE_SCHEMA)

def validate_evidence_package(pkg):
    errors = sorted(EVIDENCE_VALIDATOR.iter_errors(pkg), key=lambda e: list(e.path))
    if errors:
        msg = "\n".join([f"{list(e.path)}: {e.message}" for e in errors[:20]])
        raise AssertionError("Evidence schema validation failed:\n" + msg)
    return True

print("✅ Exact Phase-1 Evidence Package v1 schema loaded")

In [ ]:
# ============================================================
# CELL 4 — BASELINE / STATIC ARTIFACT AUDIT
# Safe to run now.
# ============================================================

for p in [
    INVENTORY_V11, SCENARIO_PATH, POLICY_PATH,
    CAPTURE_PROFILE_PATH, REQUIRED_VIEW_SET_PATH,
    CANONICAL_DIR / "demo__good__000__ai_evidence.json",
    CANONICAL_DIR / "demo__scratch__022__ai_evidence.json",
    FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__heatmap.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__overlay.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__heatmap.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__overlay.png",
]:
    assert p.exists(), f"Missing: {p}"

inventory = load_json(INVENTORY_V11)
scenario = load_json(SCENARIO_PATH)
policy = load_json(POLICY_PATH)

terms = inventory["contract_trade_terms_mapping"]

assert terms["tradeId"] == TRADE_ID
assert int(terms["policyVersion"]) == POLICY_VERSION
assert terms["policyHash"] == POLICY_HASH
assert terms["modelBundleHash"] == MODEL_BUNDLE_HASH
assert terms["captureProfileHash"] == CAPTURE_PROFILE_HASH
assert terms["productSpecHash"] == PRODUCT_SPEC_HASH
assert terms["requiredViewSetHash"] == REQUIRED_VIEW_SET_HASH
assert terms["scoreSchemaId"] == SCORE_SCHEMA_ID
assert int(terms["thresholdScaled"]) == THRESHOLD_SCALED
assert int(terms["scoreScaleExponent"]) == SCORE_SCALE_EXPONENT
assert int(terms["scoreMinScaled"]) == SCORE_MIN_SCALED
assert int(terms["scoreMaxScaled"]) == SCORE_MAX_SCALED

assert scenario["scenarioId"] == "DISPUTE_SUPPLIER_PASS_BUYER_REVIEW"
assert scenario["supplierInspection"]["datasetRelativePath"] == SUPPLIER_SOURCE
assert scenario["supplierInspection"]["aggregateScoreScaled"] == SUPPLIER_SCORE_SCALED
assert scenario["buyerInspection"]["datasetRelativePath"] == BUYER_SOURCE
assert scenario["buyerInspection"]["aggregateScoreScaled"] == BUYER_SCORE_SCALED

# Existing Supplier specimen manifest must remain the exact frozen artifact.
SUPPLIER_SPECIMEN_PATH = SUPPLIER_DIR / "SUPPLIER_SPECIMEN_MANIFEST_V1.json"
assert SUPPLIER_SPECIMEN_PATH.exists(), "Run/retain the already completed 13Y-A artifact."
supplier_specimen = load_json(SUPPLIER_SPECIMEN_PATH)
supplier_specimen_hash = jcs_keccak(supplier_specimen)

assert supplier_specimen_hash == inventory["supplier_attempt_1"]["specimenManifestHash"]
assert supplier_specimen["captureSession"]["sessionNonce"] == inventory["supplier_attempt_1"]["sessionNonce"]

# Canonical AI evidence checks
supplier_ai = load_json(CANONICAL_DIR / "demo__good__000__ai_evidence.json")
buyer_ai = load_json(CANONICAL_DIR / "demo__scratch__022__ai_evidence.json")

assert supplier_ai["inspection"]["dataset_relative_path"] == SUPPLIER_SOURCE
assert buyer_ai["inspection"]["dataset_relative_path"] == BUYER_SOURCE
assert decimal_to_scaled(supplier_ai["inspection"]["anomaly_score"]) == SUPPLIER_SCORE_SCALED
assert decimal_to_scaled(buyer_ai["inspection"]["anomaly_score"]) == BUYER_SCORE_SCALED
assert decimal_to_scaled(supplier_ai["inference_policy"]["threshold"]) == THRESHOLD_SCALED
assert decimal_to_scaled(buyer_ai["inference_policy"]["threshold"]) == THRESHOLD_SCALED

assert SUPPLIER_SCORE_SCALED < THRESHOLD_SCALED
assert BUYER_SCORE_SCALED >= THRESHOLD_SCALED

print("========== STATIC BASELINE ==========")
print("tradeId:", TRADE_ID)
print("policyHash:", POLICY_HASH)
print("modelBundleHash:", MODEL_BUNDLE_HASH)
print("Supplier:", SUPPLIER_SOURCE, SUPPLIER_SCORE_SCALED, "PASS")
print("Buyer   :", BUYER_SOURCE, BUYER_SCORE_SCALED, "REVIEW_REQUIRED")
print("Supplier specimenManifestHash:", supplier_specimen_hash)
print("✅ Static baseline PASS")

In [ ]:
# ============================================================
# CELL 5 — QA MAPPING + BUYER SPECIMEN MANIFEST + STATIC EVIDENCE
# Safe to run now.
#
# Important:
# Phase-1 schema requires focus/blur/saturation booleans.
# Public MVTec source images have no physical-camera QA telemetry.
# We therefore keep a transparent compatibility record:
# the booleans mean "no failure under the frozen public-dataset
# demo profile", NOT independently measured optical quantities.
# ============================================================

QA_COMPAT_PATH = POLICY_DIR / "EVIDENCE_QA_SCHEMA_COMPATIBILITY_V1.json"

qa_compat = {
    "project": "InspectEscrow",
    "schema": "inspectescrow.evidence-qa-schema-compatibility/1.0.0",
    "status": "FROZEN_FOR_PUBLIC_DATASET_SIMULATION",
    "scope": "PUBLIC_DATASET_SIMULATION",
    "phase1EvidenceSchema": "inspectescrow.evidence-package/1.0.0",
    "captureProfileHash": CAPTURE_PROFILE_HASH,
    "requiredViewSetHash": REQUIRED_VIEW_SET_HASH,
    "mapping": {
        "inspectionValidity": "ADMISSIBLE",
        "invalidReason": "NONE",
        "profileConforming": True,
        "requiredViewsPresent": True,
        "focusPass": True,
        "blurPass": True,
        "saturationPass": True
    },
    "semanticBoundary": {
        "focusMeasuredIndependently": False,
        "blurMeasuredIndependently": False,
        "saturationMeasuredIndependently": False,
        "booleanMeaning": "NO_FAILURE_UNDER_FROZEN_PUBLIC_DATASET_DEMO_PROFILE",
        "physicalCameraQaClaimed": False,
        "productionOpticalQaClaimed": False
    }
}
save_json(QA_COMPAT_PATH, qa_compat)

CAPTURE_QA = copy.deepcopy(qa_compat["mapping"])

def build_specimen_manifest(role_name, relative_path, out_dir, filename, existing_nonce=None):
    member, src = source_bytes_from_zip(relative_path)
    src_sha = sha256_bytes(src)
    src_keccak = keccak_bytes(src)

    with Image.open(io.BytesIO(src)) as im:
        width, height = im.size
        mode, fmt = im.mode, im.format

    assert (width, height) == (700, 700)
    assert mode == "RGB"
    assert fmt == "PNG"

    nonce = existing_nonce or ("0x" + secrets.token_hex(32))
    lot_id = "MVTec_AD_metal_nut_PUBLIC_DATASET_SIMULATION"
    specimen_id = f"dataset:{relative_path}|keccak256:{src_keccak}"

    manifest = {
        "schema": "inspectescrow.specimen-manifest",
        "schemaVersion": "1.0.0",
        "status": "FINAL_FROZEN",
        "tradeId": TRADE_ID,
        "inspectionRole": role_name,
        "attempt": 1,
        "policyHash": POLICY_HASH,
        "productSpecHash": PRODUCT_SPEC_HASH,
        "captureProfileHash": CAPTURE_PROFILE_HASH,
        "requiredViewSetHash": REQUIRED_VIEW_SET_HASH,
        "identity": {
            "scheme": "PUBLIC_DATASET_SOURCE_FILE_BINDING",
            "lotId": lot_id,
            "specimenId": specimen_id,
            "datasetRelativePath": relative_path,
            "tradeBound": True,
            "identifierVisibleOverviewRequired": False,
            "physicalSerializedIdentifierClaimed": False,
            "samePhysicalSupplierBuyerItemClaimed": False,
            "physicalCustodyProofClaimed": False
        },
        "captureSession": {
            "sessionNonce": nonce,
            "requiredViewCount": 1,
            "completeSessionFileCount": 1,
            "recaptureUsed": False
        },
        "views": [{
            "logicalName": "source_inspection_image",
            "viewId": "SOURCE_INSPECTION_IMAGE",
            "datasetRelativePath": relative_path,
            "mediaType": "image/png",
            "byteLength": len(src),
            "hashAlgorithm": "keccak256",
            "digest": src_keccak,
            "auditSha256": src_sha,
            "width": width,
            "height": height,
            "colorMode": mode,
            "isOriginal": True,
            "isOverview": True,
            "identifierBearingOverview": False
        }],
        "requiredViewsPresent": True,
        "completeSessionFileListIncluded": True,
        "physicalSubstitutionNotEliminated": True,
        "claimBoundary": (
            "Commits the exact public-dataset source bytes to this trade/session; "
            "does not prove physical origin, custody or same physical Supplier/Buyer specimen."
        )
    }

    out_path = out_dir / filename
    canonical_path = out_dir / filename.replace(".json", ".jcs.json")
    save_json(out_path, manifest)
    canonical_path.write_bytes(jcs_bytes(manifest))
    return manifest, out_path, canonical_path, jcs_keccak(manifest), member, src

# Reuse buyer nonce if notebook is rerun.
BUYER_SPECIMEN_PATH = BUYER_DIR / "BUYER_SPECIMEN_MANIFEST_V1.json"
existing_buyer_nonce = None
if BUYER_SPECIMEN_PATH.exists():
    old = load_json(BUYER_SPECIMEN_PATH)
    existing_buyer_nonce = old["captureSession"]["sessionNonce"]

buyer_specimen, BUYER_SPECIMEN_PATH, BUYER_SPECIMEN_JCS, BUYER_SPECIMEN_HASH, buyer_zip_member, buyer_src = build_specimen_manifest(
    "BUYER", BUYER_SOURCE, BUYER_DIR, "BUYER_SPECIMEN_MANIFEST_V1.json", existing_buyer_nonce
)

# Re-read supplier source bytes and static artifacts.
supplier_zip_member, supplier_src = source_bytes_from_zip(SUPPLIER_SOURCE)
supplier_heatmap = FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__heatmap.png"
supplier_overlay = FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__overlay.png"
buyer_heatmap = FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__heatmap.png"
buyer_overlay = FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__overlay.png"

# Integrity against canonical AI evidence.
assert sha256_bytes(supplier_src) == supplier_ai["inspection"]["input_image_sha256"]
assert sha256_file(supplier_heatmap) == supplier_ai["inspection"]["heatmap_sha256"]
assert sha256_file(supplier_overlay) == supplier_ai["inspection"]["overlay_sha256"]

assert sha256_bytes(buyer_src) == buyer_ai["inspection"]["input_image_sha256"]
assert sha256_file(buyer_heatmap) == buyer_ai["inspection"]["heatmap_sha256"]
assert sha256_file(buyer_overlay) == buyer_ai["inspection"]["overlay_sha256"]

BUYER_COMMITMENT = BUYER_DIR / "BUYER_SPECIMEN_MANIFEST_COMMITMENT_V1.json"
save_json(BUYER_COMMITMENT, {
    "project": "InspectEscrow",
    "schema": "inspectescrow.specimen-manifest-commitment/1.0.0",
    "status": "FINAL_FROZEN",
    "scenarioId": "DISPUTE_SUPPLIER_PASS_BUYER_REVIEW",
    "tradeId": TRADE_ID,
    "inspectionRole": "BUYER",
    "attempt": 1,
    "sessionNonce": buyer_specimen["captureSession"]["sessionNonce"],
    "source": {
        "datasetRelativePath": BUYER_SOURCE,
        "zipMember": buyer_zip_member,
        "byteLength": len(buyer_src),
        "keccak256": keccak_bytes(buyer_src),
        "sha256": sha256_bytes(buyer_src),
        "exactOriginalBytesUsed": True,
        "resavedOrTranscoded": False
    },
    "specimenManifest": {
        "readableFile": BUYER_SPECIMEN_PATH.name,
        "canonicalFile": BUYER_SPECIMEN_JCS.name,
        "canonicalization": "RFC8785-JCS",
        "hashAlgorithm": "keccak256",
        "specimenManifestHash": BUYER_SPECIMEN_HASH,
        "selfRecomputation": "PASS"
    },
    "physicalIdentityClaim": False,
    "stationSignatureCreated": False,
    "evidencePackageHashCreated": False,
    "blockchainTransaction": False
})

print("========== STATIC PREP COMPLETE ==========")
print("QA compatibility:", QA_COMPAT_PATH)
print("Supplier specimen:", supplier_specimen_hash)
print("Buyer specimen   :", BUYER_SPECIMEN_HASH)
print("Buyer sessionNonce:", buyer_specimen["captureSession"]["sessionNonce"])
print("Supplier heatmap SHA:", sha256_file(supplier_heatmap))
print("Buyer heatmap SHA   :", sha256_file(buyer_heatmap))
print("✅ Buyer specimen + all static AI evidence integrity PASS")

In [ ]:
# ============================================================
# CELL 5A — FINAL MODEL BUNDLE INTEGRITY AUDIT
#
# READ ONLY
# - exact 10 child artifacts resolved by byteLength + keccak256
# - current bundle self-hash recomputed
# - calibration link checked
# - historical supersession reference allowed only at the exact
#   semantic field $.supersedesModelBundleHash
# ============================================================

MODEL_DIR = V2_DIR / "MODEL_BUNDLE"
FINAL_MODEL_MANIFEST = MODEL_DIR / "FINAL_MODEL_BUNDLE_PHASE1_SEMANTIC_PATCH1.json"
OLD_MODEL_HASH = "0x5283badc2a43c964cda6f7a921117340f0b1928027e05c3f7aea0dcba9849466"
NEW_MODEL_HASH = MODEL_BUNDLE_HASH

assert FINAL_MODEL_MANIFEST.exists(), FINAL_MODEL_MANIFEST
model_manifest = load_json(FINAL_MODEL_MANIFEST)
artifacts = model_manifest["artifacts"]
assert len(artifacts) == 10

# Self-hash: ZERO_BUNDLE_HASH_PREIMAGE_V1
preimage = copy.deepcopy(model_manifest)
assert preimage["bundleHash"].lower() == NEW_MODEL_HASH.lower()
preimage["bundleHash"] = ZERO_BYTES32
recomputed = jcs_keccak(preimage)
assert recomputed.lower() == NEW_MODEL_HASH.lower()

# Build index of all V2 files by exact byte length.
size_index = {}
for p in V2_DIR.rglob("*"):
    if p.is_file() and p.resolve() != FINAL_MODEL_MANIFEST.resolve():
        size_index.setdefault(p.stat().st_size, []).append(p)

resolved = {}
for a in artifacts:
    expected_len = int(a["byteLength"])
    expected_digest = a["digest"].lower()
    assert a["hashAlgorithm"] == "keccak256"
    matches = []
    for p in size_index.get(expected_len, []):
        try:
            if keccak_file(p).lower() == expected_digest:
                matches.append(p)
        except Exception:
            pass
    assert len(matches) == 1, (
        f'{a["logicalName"]}: expected exactly one exact artifact match, got {len(matches)}'
    )
    resolved[a["logicalName"]] = matches[0]

digest_by_name = {a["logicalName"]: a["digest"].lower() for a in artifacts}
assert (
    model_manifest["calibrationManifestHash"].lower()
    == digest_by_name["calibration_manifest"]
)

# Stale-hash semantics:
# The old bundle hash is allowed only as a historical supersession pointer
# in MODEL_SCORE_SEMANTICS_CLARIFICATION_V1.json.
historical_hits = []
for logical_name, p in resolved.items():
    if p.suffix.lower() not in {".json",".txt",".md",".yaml",".yml",".toml",".csv",".lock"}:
        continue
    text_data = p.read_text(encoding="utf-8", errors="replace")
    if OLD_MODEL_HASH.lower() in text_data.lower():
        if p.name == "MODEL_SCORE_SEMANTICS_CLARIFICATION_V1.json":
            d = json.loads(text_data)
            assert d.get("supersedesModelBundleHash","").lower() == OLD_MODEL_HASH.lower()
            # Ensure it is not simultaneously presented as current/active bundle.
            for forbidden_key in ["modelBundleHash","currentBundleHash","activeBundleHash","effectiveBundleHash"]:
                assert str(d.get(forbidden_key,"")).lower() != OLD_MODEL_HASH.lower()
            historical_hits.append("$.supersedesModelBundleHash")
        else:
            raise AssertionError(f"Unexpected OLD modelBundleHash in committed child: {p}")

assert historical_hits == ["$.supersedesModelBundleHash"]

print("========== FINAL MODEL BUNDLE AUDIT ==========")
print("Resolved artifacts:", len(resolved), "/ 10")
print("Bundle self-hash: PASS")
print("Calibration internal link: PASS")
print("OLD hash occurrence:", historical_hits[0], "-> EXPECTED HISTORICAL SUPERSESSION")
print("Current modelBundleHash:", NEW_MODEL_HASH)
print("✅ PASS_MODEL_BUNDLE_CHILD_INTEGRITY")
print("Blockchain tx: NO")

## JIT 구간

아래 Supplier / Buyer 셀은 **실제 Station 서명과 `submitInspection()` 직전에 실행**한다.

각 셀은:
- `issuedAt` / `validUntil` 생성
- Phase-1 Evidence Package v1 조립 및 schema validation
- `ZERO_ATTESTATION_DIGEST_PREIMAGE_V1`로 `evidencePackageHash` 계산
- 정확한 EIP-712 StationAttestation digest 계산
- 최종 Evidence Package에 `attestationDigest` 삽입
- 외부 지갑/팀원이 서명할 unsigned typed-data JSON 생성

을 한다.

**Private key는 사용하지 않는다.**

In [ ]:
# ============================================================
# CELL 6 — COMMON JIT EVIDENCE / EIP-712 FUNCTIONS
# Safe helper definitions; no timestamps are frozen yet.
# ============================================================

EIP712_NAME = "InspectEscrow"
EIP712_VERSION = "1"

STATION_TYPE_STRING = (
    "StationAttestation(bytes32 tradeId,uint8 role,address station,uint32 attempt,"
    "bytes32 sessionNonce,bytes32 policyHash,bytes32 modelBundleHash,"
    "bytes32 captureProfileHash,bytes32 specimenManifestHash,bytes32 evidencePackageHash,"
    "uint128 aggregateScoreScaled,bytes32 scoreSchemaId,uint8 inspectionValidity,"
    "uint8 invalidReason,uint64 issuedAt,uint64 validUntil)"
)

EIP712_DOMAIN_TYPE_STRING = "EIP712Domain(string name,string version,uint256 chainId,address verifyingContract)"

STATION_TYPEHASH = keccak(STATION_TYPE_STRING.encode())
DOMAIN_TYPEHASH = keccak(EIP712_DOMAIN_TYPE_STRING.encode())

def addr_bytes(addr):
    return bytes.fromhex(addr[2:])

def b32(v):
    assert is_bytes32(v), v
    return bytes.fromhex(v[2:])

DOMAIN_SEPARATOR = keccak(encode(
    ["bytes32", "bytes32", "bytes32", "uint256", "address"],
    [
        DOMAIN_TYPEHASH,
        keccak(EIP712_NAME.encode()),
        keccak(EIP712_VERSION.encode()),
        CHAIN_ID,
        CONTRACT,
    ]
))

def compute_station_eip712_digest(a):
    struct_hash = keccak(encode(
        [
            "bytes32","uint8","address","uint32","bytes32","bytes32","bytes32","bytes32",
            "bytes32","bytes32","uint128","bytes32","uint8","uint8","uint64","uint64"
        ],
        [
            STATION_TYPEHASH,
            a["role"],
            a["station"],
            a["attempt"],
            b32(a["sessionNonce"]),
            b32(a["policyHash"]),
            b32(a["modelBundleHash"]),
            b32(a["captureProfileHash"]),
            b32(a["specimenManifestHash"]),
            b32(a["evidencePackageHash"]),
            a["aggregateScoreScaled"],
            b32(a["scoreSchemaId"]),
            a["inspectionValidity"],
            a["invalidReason"],
            a["issuedAt"],
            a["validUntil"],
        ]
    ))
    return "0x" + keccak(b"\x19\x01" + DOMAIN_SEPARATOR + struct_hash).hex()

def build_evidence_package(
    role_name, role_enum, station, specimen, specimen_hash,
    ai_json_path, source_bytes, heatmap_path, overlay_path,
    aggregate_scaled, out_dir, validity_seconds=1800
):
    issued_at = int(time.time())
    valid_until = issued_at + int(validity_seconds)

    ai = load_json(ai_json_path)
    source_rel = ai["inspection"]["dataset_relative_path"]

    package_id = f"InspectEscrow:{TRADE_ID}:{role_name}:attempt1:{specimen['captureSession']['sessionNonce'][2:14]}"

    pkg = {
        "schema": "inspectescrow.evidence-package",
        "schemaVersion": "1.0.0",
        "packageId": package_id,
        "tradeId": TRADE_ID,
        "inspectionRole": role_name,
        "attempt": 1,
        "sessionNonce": specimen["captureSession"]["sessionNonce"],
        "policyHash": POLICY_HASH,
        "modelBundleHash": MODEL_BUNDLE_HASH,
        "captureProfileHash": CAPTURE_PROFILE_HASH,
        "specimen": {
            "specimenId": specimen["identity"]["specimenId"],
            "lotId": specimen["identity"]["lotId"],
            "tradeBound": True,
            "overviewFileLogicalName": "source_inspection_image",
            "requiredViewSetHash": REQUIRED_VIEW_SET_HASH,
            "specimenManifestHash": specimen_hash,
            "physicalSubstitutionNotEliminated": True
        },
        "files": [{
            "logicalName": "source_inspection_image",
            "viewId": "SOURCE_INSPECTION_IMAGE",
            "mediaType": "image/png",
            "byteLength": len(source_bytes),
            "hashAlgorithm": "keccak256",
            "digest": keccak_bytes(source_bytes),
            "isOriginal": True,
            "isOverview": True
        }],
        "captureQa": copy.deepcopy(CAPTURE_QA),
        "score": {
            "scoreSchemaId": SCORE_SCHEMA_ID,
            "scaleExponent": SCORE_SCALE_EXPONENT,
            "perViewScaled": [aggregate_scaled],
            "aggregateScoreScaled": aggregate_scaled,
            "stationVerdictAuthoritative": False
        },
        "station": {
            "signer": station,
            "signatureScheme": "EIP712",
            "attestationDigest": ZERO_BYTES32,
            "issuedAt": issued_at,
            "validUntil": valid_until
        },
        "createdAt": utc_iso(issued_at),
        "heatmapFiles": [{
            "logicalName": f"{role_name.lower()}_heatmap",
            "mediaType": "image/png",
            "byteLength": heatmap_path.stat().st_size,
            "digest": keccak_file(heatmap_path)
        }],
        "reportFiles": [
            {
                "logicalName": f"{role_name.lower()}_canonical_ai_evidence",
                "mediaType": "application/json",
                "byteLength": ai_json_path.stat().st_size,
                "digest": keccak_file(ai_json_path)
            },
            {
                "logicalName": f"{role_name.lower()}_overlay",
                "mediaType": "image/png",
                "byteLength": overlay_path.stat().st_size,
                "digest": keccak_file(overlay_path)
            }
        ]
    }

    validate_evidence_package(pkg)

    # Non-circular package hash rule:
    # station.attestationDigest is ZERO_BYTES32 in hash preimage.
    evidence_hash = jcs_keccak(pkg)

    attestation = {
        "tradeId": TRADE_ID,
        "role": role_enum,
        "station": station,
        "attempt": 1,
        "sessionNonce": specimen["captureSession"]["sessionNonce"],
        "policyHash": POLICY_HASH,
        "modelBundleHash": MODEL_BUNDLE_HASH,
        "captureProfileHash": CAPTURE_PROFILE_HASH,
        "specimenManifestHash": specimen_hash,
        "evidencePackageHash": evidence_hash,
        "aggregateScoreScaled": aggregate_scaled,
        "scoreSchemaId": SCORE_SCHEMA_ID,
        "inspectionValidity": VALIDITY_ADMISSIBLE,
        "invalidReason": INVALID_REASON_NONE,
        "issuedAt": issued_at,
        "validUntil": valid_until
    }

    attestation_digest = compute_station_eip712_digest(attestation)

    final_pkg = copy.deepcopy(pkg)
    final_pkg["station"]["attestationDigest"] = attestation_digest
    validate_evidence_package(final_pkg)

    # Verify final manifest points back to same non-circular package hash.
    verify_preimage = copy.deepcopy(final_pkg)
    verify_preimage["station"]["attestationDigest"] = ZERO_BYTES32
    assert jcs_keccak(verify_preimage) == evidence_hash

    typed_data = {
        "domain": {
            "name": EIP712_NAME,
            "version": EIP712_VERSION,
            "chainId": CHAIN_ID,
            "verifyingContract": CONTRACT
        },
        "primaryType": "StationAttestation",
        "types": {
            "StationAttestation": [
                {"name":"tradeId","type":"bytes32"},
                {"name":"role","type":"uint8"},
                {"name":"station","type":"address"},
                {"name":"attempt","type":"uint32"},
                {"name":"sessionNonce","type":"bytes32"},
                {"name":"policyHash","type":"bytes32"},
                {"name":"modelBundleHash","type":"bytes32"},
                {"name":"captureProfileHash","type":"bytes32"},
                {"name":"specimenManifestHash","type":"bytes32"},
                {"name":"evidencePackageHash","type":"bytes32"},
                {"name":"aggregateScoreScaled","type":"uint128"},
                {"name":"scoreSchemaId","type":"bytes32"},
                {"name":"inspectionValidity","type":"uint8"},
                {"name":"invalidReason","type":"uint8"},
                {"name":"issuedAt","type":"uint64"},
                {"name":"validUntil","type":"uint64"},
            ]
        },
        "message": attestation
    }

    readable = out_dir / f"{role_name}_EVIDENCE_PACKAGE_V1.json"
    canonical = out_dir / f"{role_name}_EVIDENCE_PACKAGE_V1.jcs.json"
    payload = out_dir / f"{role_name}_STATION_ATTESTATION_UNSIGNED_V1.json"
    commitment = out_dir / f"{role_name}_EVIDENCE_COMMITMENT_V1.json"

    save_json(readable, final_pkg)
    canonical.write_bytes(jcs_bytes(final_pkg))
    save_json(payload, typed_data)
    save_json(commitment, {
        "project": "InspectEscrow",
        "status": "READY_FOR_EXTERNAL_STATION_SIGNATURE",
        "tradeId": TRADE_ID,
        "role": role_name,
        "specimenManifestHash": specimen_hash,
        "evidencePackageHash": evidence_hash,
        "stationAttestationDigest": attestation_digest,
        "issuedAt": issued_at,
        "validUntil": valid_until,
        "packageHashRule": "ZERO_ATTESTATION_DIGEST_PREIMAGE_V1",
        "schemaValidation": "PASS",
        "privateKeyUsed": False,
        "signatureCreated": False,
        "blockchainTransaction": False
    })

    return {
        "evidencePackageHash": evidence_hash,
        "attestationDigest": attestation_digest,
        "issuedAt": issued_at,
        "validUntil": valid_until,
        "package": final_pkg,
        "attestation": attestation,
        "typedData": typed_data,
        "readablePath": readable,
        "canonicalPath": canonical,
        "payloadPath": payload,
        "commitmentPath": commitment
    }

print("✅ JIT package/EIP-712 functions ready")

In [ ]:
# ============================================================
# CELL 7 — SUPPLIER JIT FINALIZATION
#
# RUN THIS ONLY RIGHT BEFORE Supplier Station signing /
# Supplier submitInspection().
# Default validity: 30 minutes.
# ============================================================

inventory = load_json(INVENTORY_V11)
terms = inventory["contract_trade_terms_mapping"]
SUPPLIER_STATION = terms["supplierStation"]

supplier_specimen = load_json(SUPPLIER_SPECIMEN_PATH)
supplier_ai_path = CANONICAL_DIR / "demo__good__000__ai_evidence.json"
supplier_heatmap = FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__heatmap.png"
supplier_overlay = FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__overlay.png"
_, supplier_src = source_bytes_from_zip(SUPPLIER_SOURCE)

SUPPLIER_JIT = build_evidence_package(
    role_name="SUPPLIER",
    role_enum=ROLE_SUPPLIER,
    station=SUPPLIER_STATION,
    specimen=supplier_specimen,
    specimen_hash=supplier_specimen_hash,
    ai_json_path=supplier_ai_path,
    source_bytes=supplier_src,
    heatmap_path=supplier_heatmap,
    overlay_path=supplier_overlay,
    aggregate_scaled=SUPPLIER_SCORE_SCALED,
    out_dir=SUPPLIER_DIR,
    validity_seconds=1800,
)

print("========== SUPPLIER JIT ==========")
print("evidencePackageHash :", SUPPLIER_JIT["evidencePackageHash"])
print("attestationDigest   :", SUPPLIER_JIT["attestationDigest"])
print("issuedAt            :", SUPPLIER_JIT["issuedAt"])
print("validUntil          :", SUPPLIER_JIT["validUntil"])
print("unsigned typed data :", SUPPLIER_JIT["payloadPath"])
print("⚠️ Sign/send before validUntil. No private key was used here.")

In [ ]:
# ============================================================
# CELL 8 — BUYER JIT FINALIZATION
#
# RUN THIS ONLY AFTER Supplier PASS moves trade to AWAIT_BUYER,
# and RIGHT BEFORE Buyer Station signing / submitInspection().
# ============================================================

inventory = load_json(INVENTORY_V11)
terms = inventory["contract_trade_terms_mapping"]
BUYER_STATION = terms["buyerStation"]

buyer_specimen = load_json(BUYER_SPECIMEN_PATH)
buyer_ai_path = CANONICAL_DIR / "demo__scratch__022__ai_evidence.json"
buyer_heatmap = FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__heatmap.png"
buyer_overlay = FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__overlay.png"
_, buyer_src = source_bytes_from_zip(BUYER_SOURCE)

BUYER_JIT = build_evidence_package(
    role_name="BUYER",
    role_enum=ROLE_BUYER,
    station=BUYER_STATION,
    specimen=buyer_specimen,
    specimen_hash=BUYER_SPECIMEN_HASH,
    ai_json_path=buyer_ai_path,
    source_bytes=buyer_src,
    heatmap_path=buyer_heatmap,
    overlay_path=buyer_overlay,
    aggregate_scaled=BUYER_SCORE_SCALED,
    out_dir=BUYER_DIR,
    validity_seconds=1800,
)

print("========== BUYER JIT ==========")
print("evidencePackageHash :", BUYER_JIT["evidencePackageHash"])
print("attestationDigest   :", BUYER_JIT["attestationDigest"])
print("issuedAt            :", BUYER_JIT["issuedAt"])
print("validUntil          :", BUYER_JIT["validUntil"])
print("unsigned typed data :", BUYER_JIT["payloadPath"])
print("Expected contract outcome: REVIEW_REQUIRED -> DISPUTED_PRIMARY")
print("⚠️ Sign/send before validUntil. No private key was used here.")

In [ ]:
# ============================================================
# CELL 9 — ROBUST IMMEDIATE UNUSED TRADE-ID PREFLIGHT
#
# READ ONLY
# - retries transient Sepolia RPC failures
# - requires 2+ DISTINCT exact TradeNotFound confirmations
# - NO signing / NO transaction
# ============================================================

import requests

RPC_ENDPOINTS = [
    "https://gateway.tenderly.co/public/sepolia",
    "https://ethereum-sepolia-rpc.publicnode.com",
    "https://sepolia.drpc.org",
    "https://rpc.sepolia.org",
    "https://api.zan.top/eth-sepolia",
    "https://rpc.sepolia.ethpandaops.io",
]

EXPECTED_CHAIN_ID = 11155111
MAX_ATTEMPTS = 3

GET_TRADE_STATE_SELECTOR = "0x" + keccak(b"getTradeState(bytes32)")[:4].hex()
TRADE_NOT_FOUND_SELECTOR = "0x" + keccak(b"TradeNotFound()")[:4].hex()
CALLDATA = GET_TRADE_STATE_SELECTOR + TRADE_ID[2:].rjust(64, "0")

session = requests.Session()
session.headers.update({
    "Content-Type": "application/json",
    "User-Agent": "InspectEscrow-Sepolia-Preflight/1.0"
})

def rpc_call(url, method, params, timeout=15):
    payload = {"jsonrpc":"2.0","id":1,"method":method,"params":params}
    try:
        r = session.post(url, json=payload, timeout=timeout)
        if r.status_code != 200:
            return {"_transport_error": f"HTTP {r.status_code}: {r.text[:200]}"}
        return r.json()
    except Exception as e:
        return {"_transport_error": repr(e)}

def contains_selector(obj, selector):
    selector = selector.lower()
    if isinstance(obj, str):
        return selector in obj.lower()
    if isinstance(obj, dict):
        return any(contains_selector(v, selector) for v in obj.values())
    if isinstance(obj, list):
        return any(contains_selector(v, selector) for v in obj)
    return False

results = []
for url in RPC_ENDPOINTS:
    best = None
    for attempt in range(1, MAX_ATTEMPTS + 1):
        chain = rpc_call(url, "eth_chainId", [])
        block = rpc_call(url, "eth_blockNumber", [])
        code = rpc_call(url, "eth_getCode", [CONTRACT, "latest"])

        chain_id = None
        if "result" in chain:
            try:
                chain_id = int(chain["result"], 16)
            except Exception:
                pass

        code_value = code.get("result")
        code_ok = (
            chain_id == EXPECTED_CHAIN_ID
            and isinstance(code_value, str)
            and code_value not in ["0x", "0x0", None]
        )

        if not code_ok:
            best = {
                "url":url, "attempt":attempt, "chainId":chain_id,
                "block":None, "codeOk":False,
                "verdict":"INCONCLUSIVE",
                "detail":"RPC/network/contract-code check failed"
            }
            time.sleep(1.0)
            continue

        block_number = None
        if "result" in block:
            try:
                block_number = int(block["result"], 16)
            except Exception:
                pass

        call = rpc_call(
            url, "eth_call",
            [{"to":CONTRACT, "data":CALLDATA}, "latest"]
        )

        if "result" in call:
            best = {
                "url":url, "attempt":attempt, "chainId":chain_id,
                "block":block_number, "codeOk":True,
                "verdict":"USED_TRADE_ID", "detail":call["result"]
            }
            break
        elif contains_selector(call, TRADE_NOT_FOUND_SELECTOR):
            best = {
                "url":url, "attempt":attempt, "chainId":chain_id,
                "block":block_number, "codeOk":True,
                "verdict":"UNUSED_CONFIRMED_TRADE_NOT_FOUND",
                "detail":TRADE_NOT_FOUND_SELECTOR
            }
            break
        else:
            best = {
                "url":url, "attempt":attempt, "chainId":chain_id,
                "block":block_number, "codeOk":True,
                "verdict":"INCONCLUSIVE", "detail":call
            }
        time.sleep(1.0)
    results.append(best)

print("========== ROBUST SEPOLIA PREFLIGHT ==========")
print("Contract:", CONTRACT)
print("tradeId:", TRADE_ID)
for r in results:
    print("\n", r["url"])
    print(" attempt:", r["attempt"])
    print(" chainId:", r["chainId"])
    print(" block:", r["block"])
    print(" contract code:", "PASS" if r["codeOk"] else "FAIL")
    print(" verdict:", r["verdict"])

unused_confirmations = [
    r for r in results
    if r["verdict"] == "UNUSED_CONFIRMED_TRADE_NOT_FOUND"
]
used_confirmations = [
    r for r in results
    if r["verdict"] == "USED_TRADE_ID"
]
valid_blocks = [r["block"] for r in unused_confirmations if r["block"] is not None]

print("\n========== FINAL ==========")
print("Exact TradeNotFound confirmations:", len(unused_confirmations))
print("USED confirmations:", len(used_confirmations))
if valid_blocks:
    print("Confirmed block range:", min(valid_blocks), "->", max(valid_blocks))

assert len(used_confirmations) == 0, (
    "STOP: at least one valid RPC reports tradeId USED."
)
assert len(unused_confirmations) >= 2, (
    "Still insufficient independent confirmations. DO NOT transact yet."
)

print("\n✅ tradeId UNUSED — 2+ DISTINCT RPC CONFIRMATIONS")
print("✅ IMMEDIATE PREFLIGHT PASS")
print("Private key used: NO")
print("Blockchain tx: NO")
print("\n⚠️ proposeTrade should follow immediately after this check.")

In [ ]:
# ============================================================
# CELL 10 — REPLAY WRAPPER + UNSIGNED REPLAY ATTESTATION
#
# Run only AFTER Supplier + Buyer final Evidence Package hashes
# exist and the trade is in DISPUTED_PRIMARY.
#
# This does NOT invent an Inspector settlement decision.
# ============================================================

assert "SUPPLIER_JIT" in globals(), "Run Supplier JIT first."
assert "BUYER_JIT" in globals(), "Run Buyer JIT first."

supplier_evidence_hash = SUPPLIER_JIT["evidencePackageHash"]
buyer_evidence_hash = BUYER_JIT["evidencePackageHash"]

REPLAY_SOURCE = REPLAY_DIR / "DISPUTE_022_REPLAY_RESULT.json"
assert REPLAY_SOURCE.exists(), REPLAY_SOURCE
replay_source_json = load_json(REPLAY_SOURCE)

# Keep the executed replay artifact intact and bind it into a trade-specific wrapper.
replay_wrapper = {
    "project": "InspectEscrow",
    "schema": "inspectescrow.trade-specific-replay-report/1.0.0",
    "status": "FINAL_FOR_PRIMARY_DISPUTE_REPLAY",
    "tradeId": TRADE_ID,
    "policyHash": POLICY_HASH,
    "modelBundleHash": MODEL_BUNDLE_HASH,
    "supplierEvidenceHash": supplier_evidence_hash,
    "buyerEvidenceHash": buyer_evidence_hash,
    "submittedBuyerScoreScaled": BUYER_SCORE_SCALED,
    "replayToleranceScaled": REPLAY_TOLERANCE_SCALED,
    "requiredDecisionStability": True,
    "sourceReplayArtifact": {
        "file": REPLAY_SOURCE.name,
        "sha256": sha256_file(REPLAY_SOURCE),
        "keccak256": keccak_file(REPLAY_SOURCE),
        "separateMachineVerified": False
    },
    "replayResult": replay_source_json,
    "claimBoundary": (
        "Artifact-reload / controlled replay evidence supports execution consistency "
        "within the frozen engineering tolerance; it is not a universal or cross-machine guarantee."
    )
}

REPLAY_REPORT_PATH = REPLAY_OUT_DIR / "TRADE_SPECIFIC_REPLAY_REPORT_V1.json"
REPLAY_REPORT_JCS = REPLAY_OUT_DIR / "TRADE_SPECIFIC_REPLAY_REPORT_V1.jcs.json"
save_json(REPLAY_REPORT_PATH, replay_wrapper)
REPLAY_REPORT_JCS.write_bytes(jcs_bytes(replay_wrapper))
REPLAY_REPORT_HASH = jcs_keccak(replay_wrapper)

# Current frozen evidence says scratch022 replay is consistent.
# Do not use this boolean as a product-quality verdict.
REPLAY_CONSISTENT = True

inventory = load_json(INVENTORY_V11)
terms = inventory["contract_trade_terms_mapping"]
INSPECTOR_STATION = terms["inspectorStation"]
PRIMARY_INSPECTOR = terms["primaryInspector"]

replay_nonce = "0x" + secrets.token_hex(32)
issued_at = int(time.time())
valid_until = issued_at + 1800

replay_attestation = {
    "tradeId": TRADE_ID,
    "stage": INSPECTOR_STAGE_PRIMARY,
    "inspectorActor": PRIMARY_INSPECTOR,
    "policyHash": POLICY_HASH,
    "supplierEvidenceHash": supplier_evidence_hash,
    "buyerEvidenceHash": buyer_evidence_hash,
    "replayReportHash": REPLAY_REPORT_HASH,
    "consistent": REPLAY_CONSISTENT,
    "nonce": replay_nonce,
    "issuedAt": issued_at,
    "validUntil": valid_until
}

REPLAY_UNSIGNED_PATH = REPLAY_OUT_DIR / "PRIMARY_REPLAY_ATTESTATION_UNSIGNED_V1.json"
save_json(REPLAY_UNSIGNED_PATH, {
    "domain": {
        "name": "InspectEscrow",
        "version": "1",
        "chainId": CHAIN_ID,
        "verifyingContract": CONTRACT
    },
    "primaryType": "ReplayAttestation",
    "types": {
        "ReplayAttestation": [
            {"name":"tradeId","type":"bytes32"},
            {"name":"stage","type":"uint8"},
            {"name":"inspectorActor","type":"address"},
            {"name":"policyHash","type":"bytes32"},
            {"name":"supplierEvidenceHash","type":"bytes32"},
            {"name":"buyerEvidenceHash","type":"bytes32"},
            {"name":"replayReportHash","type":"bytes32"},
            {"name":"consistent","type":"bool"},
            {"name":"nonce","type":"bytes32"},
            {"name":"issuedAt","type":"uint64"},
            {"name":"validUntil","type":"uint64"},
        ]
    },
    "message": replay_attestation,
    "signerMustBe": INSPECTOR_STATION,
    "callerMustBe": PRIMARY_INSPECTOR,
    "privateKeyIncluded": False
})

print("========== REPLAY ==========")
print("supplierEvidenceHash:", supplier_evidence_hash)
print("buyerEvidenceHash   :", buyer_evidence_hash)
print("replayReportHash    :", REPLAY_REPORT_HASH)
print("consistent          :", REPLAY_CONSISTENT)
print("unsigned payload    :", REPLAY_UNSIGNED_PATH)
print("⚠️ consistent=True is execution consistency, NOT an automatic refund/release decision.")

In [ ]:
# ============================================================
# CELL 11 — INSPECTOR DECISION TEMPLATE
#
# This cell deliberately does NOT choose RELEASE or REFUND.
# The Inspector/business demonstration must choose the outcome
# only after the replay/inspection procedure.
# ============================================================

assert "REPLAY_REPORT_HASH" in globals(), "Run replay cell first."

INSPECTOR_DECISION_TEMPLATE_PATH = REPLAY_OUT_DIR / "PRIMARY_INSPECTOR_DECISION_TEMPLATE_V1.json"

template = {
    "project": "InspectEscrow",
    "status": "TEMPLATE_NO_SETTLEMENT_PRECOMMITMENT",
    "domain": {
        "name": "InspectEscrow",
        "version": "1",
        "chainId": CHAIN_ID,
        "verifyingContract": CONTRACT
    },
    "requiredInputsAtDecisionTime": {
        "tradeId": TRADE_ID,
        "policyHash": POLICY_HASH,
        "supplierEvidenceHash": SUPPLIER_JIT["evidencePackageHash"],
        "buyerEvidenceHash": BUYER_JIT["evidencePackageHash"],
        "replayReportHash": REPLAY_REPORT_HASH,
        "stage": INSPECTOR_STAGE_PRIMARY
    },
    "allowedDecisionEnums": {
        "RELEASE_TO_SUPPLIER": SETTLEMENT_RELEASE_TO_SUPPLIER,
        "REFUND_TO_BUYER": SETTLEMENT_REFUND_TO_BUYER
    },
    "decision": None,
    "reasonCode": None,
    "nonce": None,
    "issuedAt": None,
    "validUntil": None,
    "signerMustBe": load_json(INVENTORY_V11)["contract_trade_terms_mapping"]["inspectorStation"],
    "callerMustBe": load_json(INVENTORY_V11)["contract_trade_terms_mapping"]["primaryInspector"],
    "warning": "AI REVIEW and replay consistency do not independently decide money."
}
save_json(INSPECTOR_DECISION_TEMPLATE_PATH, template)

print("✅ Inspector decision template created:")
print(INSPECTOR_DECISION_TEMPLATE_PATH)
print("Decision remains intentionally UNSET.")

In [ ]:
# ============================================================
# CELL 12 — FINAL HANDOFF / QA / ZIP
#
# Run after static prep. If JIT artifacts exist, they are included.
# No secrets are ever included.
# ============================================================

manifest = {
    "project": "InspectEscrow",
    "schema": "inspectescrow.ai-blockchain-final-handoff/1.0.0",
    "status": "AI_E2E_HANDOFF_READY",
    "network": {"name":"Ethereum Sepolia","chainId":CHAIN_ID,"contract":CONTRACT},
    "scenario": "DISPUTE_SUPPLIER_PASS_BUYER_REVIEW",
    "frozen": {
        "tradeId": TRADE_ID,
        "policyHash": POLICY_HASH,
        "modelBundleHash": MODEL_BUNDLE_HASH,
        "captureProfileHash": CAPTURE_PROFILE_HASH,
        "productSpecHash": PRODUCT_SPEC_HASH,
        "requiredViewSetHash": REQUIRED_VIEW_SET_HASH,
        "scoreSchemaId": SCORE_SCHEMA_ID,
        "thresholdScaled": THRESHOLD_SCALED,
        "replayToleranceScaled": REPLAY_TOLERANCE_SCALED
    },
    "supplier": {
        "source": SUPPLIER_SOURCE,
        "scoreScaled": SUPPLIER_SCORE_SCALED,
        "expectedContractOutcome": "PASS",
        "specimenManifestHash": supplier_specimen_hash
    },
    "buyer": {
        "source": BUYER_SOURCE,
        "scoreScaled": BUYER_SCORE_SCALED,
        "expectedContractOutcome": "REVIEW_REQUIRED",
        "specimenManifestHash": BUYER_SPECIMEN_HASH
    },
    "security": {
        "privateKeysIncluded": False,
        "signaturesCreatedByNotebook": False,
        "transactionsSentByNotebook": False,
        "physicalTruthClaimed": False,
        "samePhysicalSupplierBuyerSpecimenClaimed": False,
        "crossMachineReplayGuaranteeClaimed": False
    }
}

if "SUPPLIER_JIT" in globals():
    manifest["supplier"]["evidencePackageHash"] = SUPPLIER_JIT["evidencePackageHash"]
    manifest["supplier"]["stationAttestationDigest"] = SUPPLIER_JIT["attestationDigest"]
else:
    manifest["supplier"]["evidencePackageHash"] = None

if "BUYER_JIT" in globals():
    manifest["buyer"]["evidencePackageHash"] = BUYER_JIT["evidencePackageHash"]
    manifest["buyer"]["stationAttestationDigest"] = BUYER_JIT["attestationDigest"]
else:
    manifest["buyer"]["evidencePackageHash"] = None

if "REPLAY_REPORT_HASH" in globals():
    manifest["replay"] = {
        "replayReportHash": REPLAY_REPORT_HASH,
        "consistent": REPLAY_CONSISTENT,
        "settlementDecisionPrecommitted": False
    }
else:
    manifest["replay"] = {"replayReportHash": None}

HANDOFF_MANIFEST = HANDOFF_DIR / "FINAL_AI_BLOCKCHAIN_HANDOFF_MANIFEST.json"
save_json(HANDOFF_MANIFEST, manifest)

# Copy compact critical artifacts.
copy_candidates = [
    POLICY_PATH,
    POLICY_DIR / "INSPECTION_POLICY_V1_1_COMMITMENT.json",
    POLICY_DIR / "BLOCKCHAIN_PROPOSE_TRADE_HANDOFF_V1.json",
    POLICY_DIR / "MEANINGFUL_ONCHAIN_DEMO_SCENARIO_V1.json",
    POLICY_DIR / "ONCHAIN_UNUSED_TRADE_ID_PREFLIGHT_V1.json",
    QA_COMPAT_PATH,
    SUPPLIER_SPECIMEN_PATH,
    BUYER_SPECIMEN_PATH,
    CANONICAL_DIR / "demo__good__000__ai_evidence.json",
    CANONICAL_DIR / "demo__scratch__022__ai_evidence.json",
    FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__heatmap.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__good__000__overlay.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__heatmap.png",
    FINAL_DEMO_EVIDENCE_DIR / "demo__scratch__022__overlay.png",
    REPLAY_DIR / "DISPUTE_022_REPLAY_RESULT.json",
    HANDOFF_MANIFEST,
]

# Include JIT outputs if already generated.
for d in [SUPPLIER_DIR, BUYER_DIR, REPLAY_OUT_DIR]:
    if d.exists():
        for p in d.glob("*.json"):
            copy_candidates.append(p)

FILES_DIR = HANDOFF_DIR / "files"
FILES_DIR.mkdir(parents=True, exist_ok=True)

copied = []
seen = set()
for src in copy_candidates:
    if not src.exists():
        continue
    key = str(src.resolve())
    if key in seen:
        continue
    seen.add(key)
    dst = FILES_DIR / src.name
    shutil.copy2(src, dst)
    copied.append({
        "file": dst.name,
        "sha256": sha256_file(dst),
        "byteLength": dst.stat().st_size
    })

save_json(HANDOFF_DIR / "FILE_INTEGRITY.json", {
    "project":"InspectEscrow",
    "files": sorted(copied, key=lambda x:x["file"])
})

# Final QA
qa = {
    "project":"InspectEscrow",
    "status":"PASS_WITH_EXTERNAL_SIGNATURE_AND_TX_STEPS_REMAINING",
    "checks":{
        "frozenPolicyHash": POLICY_HASH,
        "frozenModelBundleHash": MODEL_BUNDLE_HASH,
        "supplierScoreBelowThreshold": SUPPLIER_SCORE_SCALED < THRESHOLD_SCALED,
        "buyerScoreAtOrAboveThreshold": BUYER_SCORE_SCALED >= THRESHOLD_SCALED,
        "supplierSpecimenHashPresent": is_bytes32(supplier_specimen_hash),
        "buyerSpecimenHashPresent": is_bytes32(BUYER_SPECIMEN_HASH),
        "supplierEvidenceReady": "SUPPLIER_JIT" in globals(),
        "buyerEvidenceReady": "BUYER_JIT" in globals(),
        "replayReady": "REPLAY_REPORT_HASH" in globals(),
        "privateKeysUsed": False,
        "transactionsSent": False
    },
    "remainingExternalActions":[
        "Immediate unused-tradeId recheck then proposeTrade transaction",
        "Party/Inspector acceptances and Buyer funding/activate",
        "Supplier Station external signature + Supplier submitInspection",
        "Buyer Station external signature + Buyer submitInspection",
        "Inspector Station external replay signature + recordReplayResult",
        "Inspector decision based on agreed procedure; then settlement flow"
    ]
}
save_json(HANDOFF_DIR / "FINAL_QA.json", qa)

zip_path = V2_DIR / "InspectEscrow_AI_BLOCKCHAIN_HANDOFF_FINAL.zip"
if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for p in sorted(HANDOFF_DIR.rglob("*")):
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(HANDOFF_DIR)))

print("========== FINAL HANDOFF ==========")
print("Folder:", HANDOFF_DIR)
print("ZIP   :", zip_path)
print("ZIP SHA-256:", sha256_file(zip_path))
print("Files :", len(copied))
print("Private keys included: NO")
print("Blockchain tx sent   : NO")
print("✅ Integrated AI E2E notebook handoff packaging complete")

## 실행 순서 요약

### 지금 바로
`CELL 1 → 2 → 3 → 4 → 5 → 5A → 6`

여기까지 실행하면 static preparation이 끝난다.

### 실제 데모 트랜잭션 시점
1. `CELL 5A` — Model Bundle 최종 child integrity PASS 확인
2. `CELL 9` — `tradeId` 미사용 여부 즉시 재확인
3. 팀원 Blockchain flow: `proposeTrade → accept → fund → activate`
4. `CELL 7` — Supplier JIT → 외부 Station 서명 → Supplier `submitInspection`
5. 상태가 `AWAIT_BUYER`가 된 후 `CELL 8` — Buyer JIT → 외부 Station 서명 → Buyer `submitInspection`
6. `DISPUTED_PRIMARY` 진입 후 `CELL 10` — replay payload → 외부 Inspector Station 서명 → `recordReplayResult`
7. `CELL 11` — Inspector decision template. **결정은 AI가 자동 선택하지 않는다.**
8. `CELL 12` — 최종 handoff/QA/ZIP

### 결과적으로 이 노트북이 하지 않는 것
- private key 보관
- 자동 서명
- 자동 온체인 거래
- physical truth 증명
- AI가 최종 정산 결정

이 경계를 지키면서도, AI 산출물 → evidence commitment → EIP-712 payload → blockchain handoff까지 한 파일에서 관리할 수 있다.